In [ ]:
from huggingface_hub import login

login("")

print("Login completato con successo.")

Login completato con successo.


In [6]:
# %%
import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import warnings
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (balanced_accuracy_score, f1_score, classification_report,
                             confusion_matrix, matthews_corrcoef, brier_score_loss,
                             ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve
from transformers import AutoModel, AutoConfig

warnings.filterwarnings("ignore")

# --- 1. Project Setup & Config ---
def find_project_root(name='Bambino', start=None):
    if start is None: start = os.getcwd()
    parent_dir = start
    while True:
        if os.path.basename(parent_dir) == name: return parent_dir
        new_parent = os.path.dirname(parent_dir)
        if new_parent == parent_dir: return None
        parent_dir = new_parent

PROJECT_ROOT = find_project_root('Bambino')
if PROJECT_ROOT: sys.path.append(PROJECT_ROOT)

from config import settings
from DataUtils.BoaOpenFaceDataset import BoaOpenFaceDataset

# Config
OUT_DIR = Path("./results_MOMENT")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_TYPE = 'augmented_normalized'
MOMENT_MODEL_ID = "AutonLab/MOMENT-1-large"
TARGET_LEN = 512 # Standard length for MOMENT
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print(f"Device: {DEVICE}")
print(f"Output Dir: {OUT_DIR}")

# %% --- 2. Data Loading & Resizing ---

def resize_time_series(X, target_len=512):
    """
    Resizes [L_orig, Channels] -> [Channels, 512]
    Using linear interpolation to preserve temporal structure.
    """
    L_orig, C = X.shape
    X_resampled = np.zeros((C, target_len), dtype=np.float32)
    
    original_indices = np.linspace(0, L_orig - 1, num=L_orig)
    target_indices = np.linspace(0, L_orig - 1, num=target_len)
    
    for c in range(C):
        # Interp requires 1D arrays
        X_resampled[c, :] = np.interp(target_indices, original_indices, X[:, c])
        
    return X_resampled

def prepare_dataset_moment(dataset):
    """
    Converts Dataset -> (X_tensor, Metadata_DataFrame)
    X_tensor shape: [N, C, 512]
    """
    X_list = []
    meta_rows = []
    
    # We define modalities order explicitly
    modalities = ['g', 'h', 'f'] 
    
    print(f"Processing {len(dataset)} samples...")
    
    # Iterate safely
    limit = len(dataset)
    for i in range(limit):
        try:
            # Try getting item
            X_dict, y, _ = dataset[i]
            
            # 1. Stack Modalities: [T, C]
            # Gaze(8) + Head(13) + Face(17)
            raw_data = torch.cat([X_dict['g'], X_dict['h'], X_dict['f']], dim=1).numpy()
            
            # 2. Resize to 512 & Transpose to [C, 512]
            resized_data = resize_time_series(raw_data, TARGET_LEN)
            X_list.append(resized_data)
            
            # 3. Metadata
            if hasattr(dataset, 'instances'):
                inst = dataset.instances[i]
                pt_id = inst.pt_id
                age = float(inst.age) if inst.age is not None else np.nan
                sex = int(inst.sex) if hasattr(inst, 'sex') else 0
            else:
                pt_id = f"sample_{i}"
                age = 0.0
                sex = 0
            
            meta_rows.append({
                'pt_id': pt_id,
                'age': age,
                'sex': sex,
                'label': int(y.item())
            })
            
        except Exception as e:
            print(f"Skipping index {i}: {e}")
            continue

    # Stack into Tensor: [N, C, 512]
    X_arr = np.stack(X_list, axis=0)
    
    # Handle NaNs if any (MOMENT hates NaNs)
    X_arr = np.nan_to_num(X_arr)
    
    return X_arr, pd.DataFrame(meta_rows)

print("\n📂 Loading Datasets...")
train_path = settings.get_dataset_path(DATASET_TYPE, settings.training_filename)
val_path = settings.get_dataset_path(DATASET_TYPE, settings.validation_filename)
test_path = settings.get_dataset_path(DATASET_TYPE, settings.test_filename)

train_ds = BoaOpenFaceDataset.load_dataset(train_path)
val_ds = BoaOpenFaceDataset.load_dataset(val_path)
test_ds = BoaOpenFaceDataset.load_dataset(test_path)

# Force fix stats if missing
for ds in [train_ds, val_ds, test_ds]:
    if getattr(ds, 'trial_id_stats', None) is None:
        ds.trial_id_stats = (0.0, 1.0)
    ds.modalities = ['g', 'h', 'f']

print("   Resizing Train...")
X_train_raw, df_train = prepare_dataset_moment(train_ds)
print("   Resizing Val...")
X_val_raw, df_val = prepare_dataset_moment(val_ds)
print("   Resizing Test...")
X_test_raw, df_test = prepare_dataset_moment(test_ds)

# %% --- 3. MOMENT Feature Extraction ---

print(f"\n🧠 Loading MOMENT ({MOMENT_MODEL_ID})...")
try:
    # Direct AutoModel load - Most Robust Method
    moment_model = AutoModel.from_pretrained(
        MOMENT_MODEL_ID, 
        trust_remote_code=True,
        revision="main" 
    ).to(DEVICE)
    
    # We only need the encoder part usually, but let's use the full model forward
    moment_model.eval()
    print("✅ MOMENT Loaded Successfully.")
    
except Exception as e:
    print(f"❌ FATAL: Failed to load MOMENT. Error: {e}")
    sys.exit(1)

def extract_embeddings(X_data, batch_size=4):
    """
    X_data: [N, C, 512]
    Returns: [N, Hidden_Dim] (Pooled embeddings)
    """
    embeddings = []
    
    # Create dummy mask (1 = observe everything)
    # Shape: [Batch, C, 512] -> mapped to patch tokens internally
    
    with torch.no_grad():
        for i in tqdm(range(0, len(X_data), batch_size), desc="Extracting"):
            batch_np = X_data[i:i+batch_size]
            batch_tensor = torch.tensor(batch_np, dtype=torch.float32).to(DEVICE)
            
            # Forward Pass
            # MOMENT automatically handles the patching and masking if not provided
            # We just pass input_values
            try:
                output = moment_model(x_enc=batch_tensor)
            except:
                # Fallback for some model versions that expect different arg names
                output = moment_model(batch_tensor)

            # Output handling
            # output.embeddings is usually [Batch, n_patches, hidden_dim]
            if hasattr(output, 'embeddings'):
                hidden = output.embeddings
            elif hasattr(output, 'last_hidden_state'):
                hidden = output.last_hidden_state
            else:
                hidden = output[0] # Tuple fallback
                
            # POOLING STRATEGY:
            # 1. Mean Pool over Time/Patches -> [Batch, Hidden]
            # This condenses the whole 20s clip into one vector
            pooled = hidden.mean(dim=1) 
            
            embeddings.append(pooled.cpu().numpy())
            
    return np.concatenate(embeddings, axis=0)

print("\n--- Generating Embeddings ---")
emb_train = extract_embeddings(X_train_raw)
emb_val   = extract_embeddings(X_val_raw)
emb_test  = extract_embeddings(X_test_raw)

print(f"Embedding Shape: {emb_train.shape}") # Should be [N, 1024]

# %% --- 4. Prepare for Classifier (Merge with Metadata) ---

print("\nPreparing Final Feature Matrix...")

# Metadata Processing
# We will concatenate: [Embedding (1024) | Age (1) | Sex (1)]
# We normalize Age. Sex is 0/1 integer (HGB handles it fine).

scaler_age = StandardScaler()
age_train = scaler_age.fit_transform(df_train[['age']].values)
age_val   = scaler_age.transform(df_val[['age']].values)
age_test  = scaler_age.transform(df_test[['age']].values)

# Sex
sex_train = df_train[['sex']].values
sex_val   = df_val[['sex']].values
sex_test  = df_test[['sex']].values

# Stack
X_train_final = np.hstack([emb_train, age_train, sex_train])
X_val_final   = np.hstack([emb_val,   age_val,   sex_val])
X_test_final  = np.hstack([emb_test,  age_test,  sex_test])

y_train = df_train['label'].values
y_val   = df_val['label'].values
y_test  = df_test['label'].values

print(f"Final Feature Matrix: {X_train_final.shape}")

# %% --- 5. Train HistGradientBoosting ---

print("\n🌳 Training HistGradientBoosting...")

# HGB is the best sklearn classifier for this. 
# It handles dense data + categorical features implicitly if ints.
clf = HistGradientBoostingClassifier(
    max_iter=1000,
    learning_rate=0.01,
    max_depth=5,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
    class_weight='balanced' # Crucial for 80/20 split
)

clf.fit(X_train_final, y_train)

# %% --- 6. Evaluation ---

print("\n🔎 Optimizing Threshold on Validation...")
probs_val = clf.predict_proba(X_val_final)[:, 1]

best_thr = 0.5
best_bal = 0.0
for thr in np.linspace(0.1, 0.9, 100):
    p = (probs_val >= thr).astype(int)
    bal = balanced_accuracy_score(y_val, p)
    if bal > best_bal:
        best_bal = bal
        best_thr = thr

print(f"Best Threshold: {best_thr:.2f} (Val BalAcc: {best_bal:.4f})")

print("\n📝 Final Test Evaluation...")
probs_test = clf.predict_proba(X_test_final)[:, 1]
preds_test = (probs_test >= best_thr).astype(int)

# Metrics
bal_acc = balanced_accuracy_score(y_test, preds_test)
mcc = matthews_corrcoef(y_test, preds_test)
brier = brier_score_loss(y_test, probs_test)

print(f"Test Balanced Acc: {bal_acc:.4f}")
print(f"Test MCC: {mcc:.4f}")
print(f"Brier Score: {brier:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds_test, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, preds_test)
plt.figure(figsize=(6,5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Control", "Stimulus"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"MOMENT + HGB Confusion Matrix")
plt.savefig(OUT_DIR / "moment_confusion_matrix.png")
plt.show()

# Calibration Curve
frac_pos, mean_pred = calibration_curve(y_test, probs_test, n_bins=10)
plt.figure(figsize=(6,6))
plt.plot(mean_pred, frac_pos, marker='o', label=f'Model (Brier={brier:.3f})')
plt.plot([0,1], [0,1], '--', color='gray')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve')
plt.legend()
plt.savefig(OUT_DIR / "moment_calibration.png")
plt.show()

# %% --- 7. Per-Baby Analysis ---
print("\n👶 Per-Baby Analysis...")

df_test['pred_label'] = preds_test
df_test['true_label'] = y_test

pt_ids = df_test['pt_id'].unique()
per_baby_details = []

for pid in pt_ids:
    sub = df_test[df_test['pt_id'] == pid]
    if len(sub) < 1: continue
    
    f1 = f1_score(sub['true_label'], sub['pred_label'], zero_division=0)
    
    age = sub['age'].iloc[0] # Original Age (unscaled)
    sex_val = sub['sex'].iloc[0]
    sex_str = "F" if str(sex_val) == "0" else "M"
    
    per_baby_details.append((pid, age, sex_str, f1))

per_baby_details.sort(key=lambda x: x[3])

print("-" * 60)
print(f"{'BabyID':<15} | {'Age':<5} | {'Sex':<3} | {'F1 Score'}")
print("-" * 60)
for pid, age, sex, f1 in per_baby_details:
    print(f"{pid:<15} | {age:<5.2f} | {sex:<3} | {f1:.4f}")
print("-" * 60)

# Scatter Plot
ages = [x[1] for x in per_baby_details]
f1s = [x[3] for x in per_baby_details]

plt.figure(figsize=(8, 6))
sns.scatterplot(x=ages, y=f1s, s=100, color='darkorange', edgecolor='black')
sns.regplot(x=ages, y=f1s, scatter=False, color='gray', line_kws={'linestyle':'--'})
plt.title("Correlation: Baby Age vs. Model F1 Score (MOMENT)")
plt.xlabel("Age (Months)")
plt.ylabel("F1 Score")
plt.grid(True, alpha=0.3)
plt.savefig(OUT_DIR / "moment_age_correlation.png")
plt.show()

print("\n✅ Done!")

Device: cuda
Output Dir: results_MOMENT

📂 Loading Datasets...
   Resizing Train...
Processing 1970 samples...
   Resizing Val...
Processing 140 samples...
   Resizing Test...
Processing 137 samples...

🧠 Loading MOMENT (AutonLab/MOMENT-1-large)...
❌ FATAL: Failed to load MOMENT. Error: Unrecognized model in AutonLab/MOMENT-1-large. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: afmoe, aimv2, aimv2_vision_model, albert, align, altclip, apertus, arcee, aria, aria_text, audio-spectrogram-transformer, audioflamingo3, audioflamingo3_encoder, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, blt, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, cohere2_vision, 

AttributeError: 'tuple' object has no attribute 'tb_frame'